In [ ]:
import numpy as np

N_ACTIONS = 4
MAX_STEPS = 2
EMB_DIM = 768

class SearchEnv:
    def __init__(self, rng=None):
        self.rng = np.random.RandomState(rng)
        self.step_count = 0

    def reset(self, query_features):
        """
        query_features: dict with
          - 'length', 'embedding', 'score_spread', 'score_entropy',
          - 'prior_coverage', 'max_prior', 'mean_prior'
        """
        self.step_count = 0
        self.state = self._build_state(query_features)
        return self.state

    def _build_state(self, qf):
        # agents_used one-hot (3 agents)
        agents_used = np.zeros(3, dtype=np.float32)
        step = np.array([0.0], dtype=np.float32)

        ndcg_change = np.array([0.0], dtype=np.float32)
        recall_change = np.array([0.0], dtype=np.float32)
        elapsed_time = np.array([0.0], dtype=np.float32)
        cost = np.array([0.0], dtype=np.float32)

        # valid_actions initially: all 1 except perhaps some rules
        valid_actions = np.ones(N_ACTIONS, dtype=np.float32)

        features = [
            np.array([qf["length"]], np.float32),
            qf["embedding"].astype(np.float32),
            np.array([qf["score_spread"], qf["score_entropy"]], np.float32),
            agents_used,
            step,
            ndcg_change, recall_change, elapsed_time, cost,
            np.array([qf["prior_coverage"],
                      qf["max_prior"],
                      qf["mean_prior"]], np.float32),
            valid_actions,
        ]
        return np.concatenate(features, axis=0)

    def step(self, action):
        """
        action: int in [0, 3]; env should ignore invalid actions or treat as no-op / penalty.
        """
        self.step_count += 1

        # unpack state slices for readability
        # (you can refactor into helper methods or use a dataclass)
        # Example layout: see build_state order
        idx = 0
        query_length = self.state[idx]; idx += 1
        embedding = self.state[idx:idx+EMB_DIM]; idx += EMB_DIM
        score_spread, score_entropy = self.state[idx:idx+2]; idx += 2
        agents_used = self.state[idx:idx+3]; idx += 3
        step_val = self.state[idx]; idx += 1
        ndcg_change = self.state[idx]; idx += 1
        recall_change = self.state[idx]; idx += 1
        elapsed_time = self.state[idx]; idx += 1
        cost = self.state[idx]; idx += 1
        prior_cov, max_prior, mean_prior = self.state[idx:idx+3]; idx += 3
        valid_actions = self.state[idx:idx+N_ACTIONS]



        # simulate dynamics: how each action changes metrics
        # You should plug in models fit on logs; below are placeholders.
        delta_ndcg, delta_recall, delta_time, delta_cost, satisfaction = \
            self._simulate_effects(action, embedding, score_spread,
                                   score_entropy, prior_cov, max_prior,
                                   mean_prior)

        ndcg_change += delta_ndcg
        recall_change += delta_recall
        elapsed_time += delta_time
        cost += delta_cost

        # update agents_used and valid_actions based on action semantics
        agents_used, valid_actions = self._update_agents_and_mask(
            agents_used, valid_actions, action
        )

        step_val = float(self.step_count)

        # reward: multi-objective
        num_actions_penalty = 1.0  # or 0 at first step, as you define
        reward = (0.4 * ndcg_change +
                  0.3 * satisfaction -
                  0.2 * cost -
                  0.1 * num_actions_penalty)

        done = (self.step_count >= MAX_STEPS)

        # rebuild state vector
        new_state = np.concatenate([
            np.array([query_length], np.float32),
            embedding.astype(np.float32),
            np.array([score_spread, score_entropy], np.float32),
            agents_used.astype(np.float32),
            np.array([step_val], np.float32),
            np.array([ndcg_change, recall_change,
                      elapsed_time, cost], np.float32),
            np.array([prior_cov, max_prior, mean_prior], np.float32),
            valid_actions.astype(np.float32),
        ])
        self.state = new_state
        info = {"satisfaction": satisfaction}

        return new_state, float(reward), done, info

    # --- Placeholder models; you replace with learned/stochastic models ---
    def _simulate_effects(self, action, embedding, score_spread,
                          score_entropy, prior_cov, max_prior, mean_prior):
        
        if action == STOP_ACTION:
        return 0.0, 0.0, 0.0, 0.0, 0.0  # No effects

        agent = get_agent(action)
        if agent is None:
            raise ValueError(f"Invalid action {action}")

        # Extract query features (persistent across steps)
        query_features = {
            'embedding': embedding,
            'score_spread': score_spread,
            'score_entropy': score_entropy,
            'prior_cov': prior_cov,
            'max_prior': max_prior,
            'mean_prior': mean_prior,
        }

        # Call real agent
        effects = agent(query_features)  # or pass full state if needed

        return (effects['delta_ndcg'], effects['delta_recall'],
                effects['delta_time'], effects['delta_cost'],
                effects['satisfaction'])

        # satisfaction proxy from priors (again: fit from logs in practice)
        satisfaction = float(prior_cov * max_prior)

        return delta_ndcg, delta_recall, delta_time, delta_cost, satisfaction

    def _update_agents_and_mask(self, agents_used, valid_actions, action):
        # Example: mark agent as used, mask repeating same agent if needed
        if action in [0, 1, 2]:
            agents_used[action] = 1.0

        # Example masking logic: once STOP (3) chosen, everything else invalid
        if action == 3:
            valid_actions[:] = 0.0
        return agents_used, valid_actions
